# F1 Prediction Model Training

This notebook trains a model to predict top-3 finishers in a race.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
import joblib
import os

## 1. Load Data

In [ ]:
races = pd.read_csv('../ml_data/races.csv')
results = pd.read_csv('../ml_data/results.csv')
drivers = pd.read_csv('../ml_data/drivers.csv')
constructors = pd.read_csv('../ml_data/constructors.csv')
circuits = pd.read_csv('../ml_data/circuits.csv')

## 2. Merge Data

In [ ]:
df = results.merge(races, on='raceId', suffixes=('_result', '_race'))
df = df.merge(drivers, on='driverId')
df = df.merge(constructors, on='constructorId')
df = df.merge(circuits, on='circuitId', suffixes=('_constructor', '_circuit'))

## 3. Feature Engineering

In [ ]:
df['top_3'] = df['position'].apply(lambda x: 1 if x <= 3 else 0)
features = ['grid', 'driverId', 'constructorId', 'circuitId', 'year']
target = 'top_3'

df = df[features + [target]].dropna()

## 4. Encode Categorical Variables

In [ ]:
encoders = {col: LabelEncoder() for col in ['driverId', 'constructorId', 'circuitId']}
for col, encoder in encoders.items():
    df[col] = encoder.fit_transform(df[col])

## 5. Train Model

In [ ]:
X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

## 6. Evaluate Model

In [ ]:
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy}')

## 7. Save Artifacts

In [ ]:
model_path = '../backend/app/models/f1_model.pkl'
encoders_path = '../backend/app/models/encoders.pkl'

os.makedirs(os.path.dirname(model_path), exist_ok=True)

joblib.dump(model, model_path)
joblib.dump(encoders, encoders_path)